## Structural comparison: AI model vs. chatbot vs. agent.

#### Understand difference in plain Python before any real model is involved.

### LLM / AI Model / Brain of Agent
* A single one-shot prediction. Takes a question, returns an answer,
* Remembers nothing about any call that happened before or after it.
* This is the whole capability of a raw model with no wrapping around it.

In [2]:
def ai_model(question: str) -> str:
    return f"[Mock prediction for]: {question}"

### Chatbot (LLM /Brain With Memory)
* Wraps ai_model() with conversation history.
* Each call to LLM / Brain appends both the question and the answer to history, So next call can see previous conversation.
* It still has no ability to check anything in the real world - only to talk, with better memory of the chat.

In [28]:
import json
class Chatbot:

    def __init__(self) -> None:
        self.history: list[dict] = []

    def ask(self, question: str) -> str:
        self.history.append({"role": "user", "content": question})
        print(F"Calling LLM with question along with history:\n", json.dumps(self.history, indent=2))
        answer = ai_model(self.history)
        self.history.append({"role": "assistant", "content": answer})
        return answer

### Calling chatbot to see history is maintained.

In [32]:
chatbot = Chatbot()
print(chatbot.ask("What's the capital of France?"))
print(chatbot.ask("What's 2 time 3 ?"))
print(chatbot.ask("What's the weather in India?"))

Calling LLM with question along with history:
 [
  {
    "role": "user",
    "content": "What's the capital of France?"
  }
]
[Mock prediction for]: [{'role': 'user', 'content': "What's the capital of France?"}]
Calling LLM with question along with history:
 [
  {
    "role": "user",
    "content": "What's the capital of France?"
  },
  {
    "role": "assistant",
    "content": "[Mock prediction for]: [{'role': 'user', 'content': \"What's the capital of France?\"}]"
  },
  {
    "role": "user",
    "content": "What's 2 time 3 ?"
  }
]
[Mock prediction for]: [{'role': 'user', 'content': "What's the capital of France?"}, {'role': 'assistant', 'content': '[Mock prediction for]: [{\'role\': \'user\', \'content\': "What\'s the capital of France?"}]'}, {'role': 'user', 'content': "What's 2 time 3 ?"}]
Calling LLM with question along with history:
 [
  {
    "role": "user",
    "content": "What's the capital of France?"
  },
  {
    "role": "assistant",
    "content": "[Mock prediction for]: 

### Agents (LLM/Brain With Memory & Addition Tools)
* Wraps ai_model() with history & a set of tools it can choose to use.
* decide_tool() here is only good enough to illustrate the idea of "the assistant picks a tool." 

In [33]:
class Agent:
    def __init__(self, tools: dict[str, callable]) -> None:
        self.history: list[dict] = []
        self.tools = tools

    def decide_tool(self, question: str) -> str | None:
        for name in self.tools:
            if name in question.lower():
                return name
        return None

    def ask(self, question: str) -> str:
        self.history.append({"role": "user", "content": question})
        tool_name = self.decide_tool(question)

        if tool_name is not None:
            result = self.tools[tool_name]()
            answer = f"[used tool: {tool_name}] {result}"
            print(f"Fetched results from {tool_name} Tool")
        else:
            answer = ai_model(question)
            print(f"Fetched results from LLM")

        self.history.append({"role": "assistant", "content": answer})
        return answer

### Defining dummy tool

In [34]:
def weather() -> str:
    print("Weather tool is called")
    return "22C, partly cloudy"

### Calling Agent with 2 queries
1. Query for which Weather tool will be called.
2. Query for which results are returned by LLM.

In [37]:
agent = Agent(tools={"weather": weather})
print(agent.ask("What's the weather like today?"), '\n')
print(agent.ask("What's the capital of France?"))

Weather tool is called
Fetched results from weather Tool
[used tool: weather] 22C, partly cloudy 

Fetched results from LLM
[Mock prediction for]: What's the capital of France?
